In [ ]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext
from llama_index.core.node_parser import CodeSplitter
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings  
from llama_index.core.node_parser import SentenceSplitter

c:\Users\joaol\diff_crew\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
Settings.embed_model = HuggingFaceEmbedding(
    model_name="intfloat/multilingual-e5-large",  
    device="cpu",
    embed_batch_size=2
)

In [3]:
exclude = [".venv", "__pycache__", ".git", "build", "dist"]

py_docs = SimpleDirectoryReader(
    "../knowledge/repos",
    recursive=True,
    required_exts=[".py"],
    exclude=exclude
).load_data()

text_docs = SimpleDirectoryReader(
    "../knowledge/repos",
    recursive=True,
    required_exts=[".md", ".yml"],
    exclude=exclude
).load_data()

py_nodes = CodeSplitter(
    language="python",
    chunk_lines=120,
    chunk_lines_overlap=20
).get_nodes_from_documents(py_docs)

text_nodes = SentenceSplitter().get_nodes_from_documents(text_docs)

nodes = py_nodes + text_nodes


In [4]:
client=chromadb.PersistentClient(path="storage/chormadb")
collections=client.get_or_create_collection(name="codes")
vector_store=ChromaVectorStore(chroma_collection=collections)
storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

2026-02-22 00:55:49,654 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


In [5]:
create_index=VectorStoreIndex(nodes=nodes, 
                            storage_context=storage_context)